In [ ]:
# 将matplotlib的图嵌入到代码中
%matplotlib inline
# 用于生成随机下降梯度
import random
import torch
from d2l import torch as d2l


In [ ]:
def synthetic_data(w,b,num_examples):
    """生成 y=Xw+b+噪声"""
    # 是一个均值为0，方差为1的随机数，生成num_examples个样本，列数为w的长度
    X = torch.normal(0,1,(num_examples,len(w)))
    # y 的公式
    y = torch.matmul(X,w)+b
    # y 再加上均值为0，方差为0.01，形状与y的长度一样的随机噪声
    y += torch.normal(0,0.01,y.shape)
    return X,y.reshape((-1,1))

true_w = torch.tensor([2,-3.4])
true_b = 4.2
features,labels = synthetic_data(true_w,true_b,1000)
print('features:',features[0],"\nlabels:",labels[0])

In [ ]:
d2l.set_figsize()
# 对图像进行绘制
d2l.plt.scatter(features[:,1].detach().numpy(),labels.detach().numpy(),1)

In [ ]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    # 生成每个样本的index
    indices = list(range(num_examples))
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i:i + batch_size])
        yield features[batch_indices], labels[batch_indices]


batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break

In [ ]:
w = torch.normal(0,0.01,size=(2,1),requires_grad=True)
b = torch.zeros(1,requires_grad=True)

In [ ]:
def linreg(X, w, b):
    """线性回归模型"""
    return torch.matmul(X,w) + b

In [ ]:
def squared_loss(y_hat, y):
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 /2

In [ ]:
def sgd(params,lr,batch_size):
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad/batch_size
            # 把梯度设置为0
            param.grad.zero_()


In [ ]:


# 学习率
lr = 0.03
num_epochs = 3
# 定义参数，用于更方便的切换模型
net = linreg
# 损失函数，用于更方便的切换
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        # X和y的小批量损失
        L = loss(net(X, w, b), y)
        # 因为L的形状是(batch_size,1)，而不是标量，L中所有元素被驾到
        # 并以此计算关于[w,b]的梯度
        L.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = loss(net(features,w,b), labels)
        print(f'epoch: {epoch + 1}, loss: {float(train_l.mean()):.8f}')
